---
numbering:
  title: false
  headings: false
---

(sec:project_spectra_classification)=

# Classifying chemical samples from spectra

## Background

Imagine you are working in the lab and want to analyse the functional groups of an unknown molecule via IR spectroscopy. If you have no idea about your molecule it may take a while to identify its functional groups. While one single molecule might not be too time-consuming, it gets more time-consuming the more molecules have to be analysed. With the help of classification and machine learning, you can save a lot of time analysing the IR spectra of your molecules. 

In this project, we used machine learning so that our system could learn to identify the different functional groups in a spectrum based on a large number of IR spectra.
To reduce the code's runtime, we applied PCA to eliminate the regions that are not needed because they contain no information in the form of peaks in any of the spectra. 
The machine learning code we’re using here is a RandomForestClassifier. This algorithm consists of many trees that analyze the training data using a sampling approach, with each tree making its own prediction. Ultimately, these predictions are then combined into a single result. 
Since we don’t just want a single final result here we also need a MultiOutputClassifier beforehand, which allows multiple results to be displayed simultaneously. This is necessary in case the imported molecule has more than one functional group.

## Assets

The IR data chunks are too large to store in this repository. Download `IR_data_chunk001_of_009.parquet` and `IR_data_chunk002_of_009.parquet` from the original Zenodo record (https://zenodo.org/records/16417648) and place them in `assets/data/projects/` before executing the notebook.

Required packages:
    https://anaconda.org/channels/conda-forge/packages/rdkit/overview
    https://anaconda.org/channels/conda-forge/packages/scikit-learn/overview

## Implementation


In [1]:
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

The spectrum data files are imported and merged into a single file.
Both of the files have 20000 Ir-spectrums.

In [2]:
df_ir_1 = pd.read_parquet("../../assets/data/projects/IR_data_chunk001_of_009.parquet")
df_ir_2 = pd.read_parquet("../../assets/data/projects/IR_data_chunk002_of_009.parquet")
df_ir = pd.concat([df_ir_1, df_ir_2], axis=0).reset_index(drop=True)

The smiles specified in the dataset are reorganized into functional groups and serve as labels (y) for machine learning.

In [3]:
from rdkit import Chem
FUNCTIONAL_GROUPS = {
    "Alcohol": "[OX2H]",
    "CarboxylicAcid": "C(=O)[OH]",
    "Ketone": "[CX3](=O)[#6]",
    "Aldehyde": "[CX3H1](=O)[#6]",
    "Amine": "[NX3;H2,H1,H0]",
    "Amide": "C(=O)N",
    "Ester": "C(=O)O[#6]",
    "Ether": "[OD2]([#6])[#6]",
    "Alkene": "C=C",
    "Alkyne": "C#C",
    "Aromatic": "a"
}

Patterns={
    name: Chem.MolFromSmarts(smarts)
    for name, smarts in FUNCTIONAL_GROUPS.items()
}

In [4]:
def detect_groups( smiles):
    mol= Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return {
        name: int(mol.HasSubstructMatch(pattern))
        for name, pattern in Patterns.items()
    }

In [5]:
labels=df_ir["smiles"].apply(detect_groups)
label_df= pd.DataFrame(labels.tolist())
df= pd.concat([df_ir, label_df], axis=1)

X is defined as the ir-spectra and, the features are reduced using PCA

In [6]:
X_1= np.stack(df_ir["ir_spectra"].values)
X_1= X_1/ X_1.max(axis=1, keepdims=True)

pca = PCA(n_components=0.95)
X=pca.fit_transform(X_1)

In [7]:
y=label_df

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.02,
    random_state=42
)

In [8]:
model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=200,   
        max_depth=20,       
        n_jobs=-1,          
        random_state=42,
        class_weight="balanced"
    )
)

model.fit(X_train, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,RandomForestC...ndom_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary <n_jobs>` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Class labels.",list,"[array([0, 1]), array([0, 1]), array([0, 1]), array([0, 1]), ...]"
estimators_ estimators_: list of ``n_output`` estimatorsEstimators used for predictions.,list,"[RandomForestC...ndom_state=42), RandomForestC...ndom_state=42), RandomForestC...ndom_state=42), RandomForestC...ndom_state=42), ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying `estimator` exposes such an attribute when fit... versionadded:: 0.24,int,1969
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'


the defined model gets tested

In [9]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, 
        target_names=list(FUNCTIONAL_GROUPS.keys()),
                           zero_division=0))

                precision    recall  f1-score   support

       Alcohol       0.75      0.18      0.29       223
CarboxylicAcid       0.92      0.15      0.26        80
        Ketone       0.81      0.79      0.80       455
      Aldehyde       1.00      0.25      0.40        20
         Amine       0.74      0.86      0.79       512
         Amide       0.75      0.64      0.69       278
         Ester       0.76      0.56      0.64       209
         Ether       0.72      0.74      0.73       430
        Alkene       1.00      0.01      0.03        77
        Alkyne       1.00      0.23      0.38        13
      Aromatic       0.93      0.95      0.94       722

     micro avg       0.80      0.71      0.76      3019
     macro avg       0.85      0.49      0.54      3019
  weighted avg       0.81      0.71      0.72      3019
   samples avg       0.81      0.72      0.73      3019



To import an Excel file containing a measured spectrum, you must edit that file.

In [10]:
def predict_functional_groups(spectrum, pca, model, threshold=0.5):
 
    spectrum = spectrum / spectrum.max()
    
    spectrum_pca = pca.transform(spectrum.reshape(1, -1))
    
    y_prob = np.array([est.predict_proba(spectrum_pca)[:, 1] 
                       for est in model.estimators_]).flatten()

    result = {}
    for name, prob in zip(FUNCTIONAL_GROUPS.keys(), y_prob):
        result[name] = {
            "vorhanden": prob >= threshold,
            "wahrscheinlichkeit": round(prob, 2)
        }
    return result

In [11]:
from scipy.interpolate import interp1d

In [12]:
test_indices = y_test.index

test_idx = test_indices[0]  

spectrum_test = X_1[test_idx]

ergebnis = predict_functional_groups(spectrum_test, pca, model, threshold=0.5)

print("Vorhergesagte funktionelle Gruppen:")
for gruppe, info in ergebnis.items():
    status = "✅ JA" if info["vorhanden"] else "❌ NEIN"
    print(f"{gruppe:15} {status}  (Wahrscheinlichkeit: {info['wahrscheinlichkeit']})")

print("\nEchte Labels:")
print(label_df.iloc[test_idx])

Vorhergesagte funktionelle Gruppen:
Alcohol         ❌ NEIN  (Wahrscheinlichkeit: 0.41)
CarboxylicAcid  ❌ NEIN  (Wahrscheinlichkeit: 0.29)
Ketone          ✅ JA  (Wahrscheinlichkeit: 0.58)
Aldehyde        ❌ NEIN  (Wahrscheinlichkeit: 0.05)
Amine           ✅ JA  (Wahrscheinlichkeit: 0.64)
Amide           ✅ JA  (Wahrscheinlichkeit: 0.69)
Ester           ❌ NEIN  (Wahrscheinlichkeit: 0.49)
Ether           ✅ JA  (Wahrscheinlichkeit: 0.54)
Alkene          ❌ NEIN  (Wahrscheinlichkeit: 0.26)
Alkyne          ❌ NEIN  (Wahrscheinlichkeit: 0.05)
Aromatic        ✅ JA  (Wahrscheinlichkeit: 0.79)

Echte Labels:
Alcohol           0
CarboxylicAcid    0
Ketone            1
Aldehyde          0
Amine             1
Amide             1
Ester             0
Ether             0
Alkene            0
Alkyne            0
Aromatic          1
Name: 32823, dtype: int64


In [15]:
training_wavenumbers=df_ir["Frequency(cm^-1)"].iloc[0]

In [16]:
def load_and_predict_spectrum(excel_path, pca, model, threshold=0.5):
    
    df_new = pd.read_excel(excel_path)

    wavenumbers_new = pd.to_numeric(df_new.iloc[:, 0], errors='coerce').values
    intensities_new = pd.to_numeric(df_new.iloc[:, 1], errors='coerce').values
    
    # NaN-Werte entfernen (falls Konvertierung fehlschlägt)
    mask = ~np.isnan(wavenumbers_new) & ~np.isnan(intensities_new)
    wavenumbers_new = wavenumbers_new[mask]
    intensities_new = intensities_new[mask]
    
 
    interpolator = interp1d(wavenumbers_new, intensities_new,
                           kind='linear',
                           fill_value='extrapolate')
    spectrum_resampled = interpolator(training_wavenumbers)
    
 
    ergebnis = predict_functional_groups(spectrum_resampled, pca, model, threshold)
    

    print("Vorhergesagte funktionelle Gruppen:")
    for gruppe, info in ergebnis.items():
        status = "✅ JA" if info["vorhanden"] else "❌ NEIN"
        print(f"{gruppe:15} {status}  (Wahrscheinlichkeit: {info['wahrscheinlichkeit']})")

In [17]:
from pathlib import Path

example_spectrum = Path("../../assets/data/projects/Vanillin.xlsx")
if example_spectrum.exists():
    load_and_predict_spectrum(example_spectrum, pca, model)
else:
    print("Optional example file Vanillin.xlsx was not found; skipping this prediction example.")


Vorhergesagte funktionelle Gruppen:
Alcohol         ❌ NEIN  (Wahrscheinlichkeit: 0.49)
CarboxylicAcid  ❌ NEIN  (Wahrscheinlichkeit: 0.23)
Ketone          ✅ JA  (Wahrscheinlichkeit: 0.53)
Aldehyde        ❌ NEIN  (Wahrscheinlichkeit: 0.25)
Amine           ✅ JA  (Wahrscheinlichkeit: 0.61)
Amide           ❌ NEIN  (Wahrscheinlichkeit: 0.36)
Ester           ❌ NEIN  (Wahrscheinlichkeit: 0.3)
Ether           ✅ JA  (Wahrscheinlichkeit: 0.54)
Alkene          ❌ NEIN  (Wahrscheinlichkeit: 0.23)
Alkyne          ❌ NEIN  (Wahrscheinlichkeit: 0.03)
Aromatic        ✅ JA  (Wahrscheinlichkeit: 0.88)


## Results
Using this code, you can now import your measured IR spectrum and have the program calculate which functional groups are present in it. To do this, the wavelength must be in the first column of the Excel file and the intensity in the second. 

## Further Questions

Possible Extensions are to use more spectral data for training, so the program gets way more precisely. Furthermore there could be more functional groups to look for in the given spectra to have more information about the (unknown) molecule. The biggest extension we thought about would have been to add NMR and MS to complete the analysis of the molecule.
